In [ ]:
import pandas as pd
df = pd.read_parquet(r"C:\Users\rasmu\Desktop\Candidate\Data\gefs\emos_ready.parquet")
df = df[df.lead_time == '0 days 03:00:00']
df['y_lag1'] = df['y'].shift(1)
df['y_lag2'] = df['y'].shift(2)

df = df.dropna()
X = df.drop(['init_time','lead_time','y'], axis=1)
y = df['y']

In [ ]:
import pandas as pd

var = pd.read_parquet(r"C:\Users\rasmu\Desktop\Candidate\Data\gefs\klga.parquet").set_index('valid_time')

In [ ]:
cloud = var[['total_cloud_mean_atmosphere']].groupby([var.index]).mean()



In [ ]:
from itertools import repeat
import time
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm


def emos(params: list, y_true, member_values: pd.DataFrame, mu_vars: pd.DataFrame):
    a = params[0]
    b_ens = params[1]
    b_cloud = params[2]
    b_lag1 = params[3]
    b_lag2 = params[4]
    c = params[-2]
    d = params[-1]
    
    cloud_mean = member_values['cloud_mean'].values
    y_lag1 = member_values['y_lag1'].values
    y_lag2 = member_values['y_lag2'].values
    
    ens_mean = member_values.drop(['cloud_mean', 'y_lag1', 'y_lag2'], axis=1).mean(axis=1).values
    mu = a + b_ens * ens_mean + b_cloud * cloud_mean + b_lag1 * y_lag1 + b_lag2 * y_lag2

    spread2 = member_values.drop(['cloud_mean', 'y_lag1', 'y_lag2'], axis=1).var(axis=1, ddof=1).values
    sigma2 = c + d * spread2
    sigma2 = np.maximum(sigma2, 1e-6)
    sigma = np.sqrt(sigma2)

    z = (y_true - mu) / sigma

    nll = 0.5 * np.log(2 * np.pi * sigma2) + ((y_true - mu) ** 2) / (2 * sigma2)

    crps = sigma * (
        z * (2 * norm.cdf(z) - 1)
        + 2 * norm.pdf(z)
        - 1 / np.sqrt(np.pi)
    )

    lam = 0.5
    loss = lam * nll + (1 - lam) * crps

    return np.mean(loss)


window_size = 40

a = 0
b = np.array([0.1, 0.1, 0.1, 0.1])
c = 1
d = 1

pred_df = pd.DataFrame(index=y.index)

# history available at each step
X_hist = X.copy()
y_hist = y.copy()

for i in range(len(X)):
    # rolling 40-row training window
    X_window = X_hist.iloc[-window_size:]
    y_window = y_hist.iloc[-window_size:]

    y_true = y_window.values
    member_values = X_window
    mu_vars = member_values

    history = []
    start_time = time.time()

    def callback(xk):
        iteration = len(history) + 1
        current_loss = emos(xk, y_true, member_values, mu_vars)
        elapsed = time.time() - start_time

        history.append(current_loss)

        print(
            f"Step {i+1:4d}/{len(X)} | "
            f"Iter {iteration:4d} | "
            f"Loss: {current_loss:.6f} | "
            f"Elapsed: {elapsed:.1f}s"
        )

    results = minimize(
        fun=emos,
        x0=[a, *b, c, d],
        args=(y_true, member_values, mu_vars),
        method="L-BFGS-B",
        callback=callback,
        options={'maxiter': 100}
    )

    best_params = results.x

    a_step = best_params[0]
    b_ens_step = best_params[1]
    b_cloud_step = best_params[2]
    b_lag1_step = best_params[3]
    b_lag2_step = best_params[4]
    c_step = best_params[-2]
    d_step = best_params[-1]

    # predict 1 row ahead
    X_next = X.iloc[[i]]
    y_next = y.iloc[[i]]

    member_values_test = X_next
    mu_vars_test = member_values_test
    y_true_test = y_next.values
    cloud_mean = member_values_test['cloud_mean'].values
    y_lag1 = member_values_test['y_lag1'].values
    y_lag2 = member_values_test['y_lag2'].values

    ens_mean = member_values_test.drop(['cloud_mean', 'y_lag1', 'y_lag2'], axis=1).mean(axis=1).values
    mu = (
        a_step
        + b_ens_step * ens_mean
        + b_cloud_step * cloud_mean
        + b_lag1_step * y_lag1
        + b_lag2_step * y_lag2
    )

    spread2 = member_values_test.drop(['cloud_mean', 'y_lag1', 'y_lag2'], axis=1).var(axis=1, ddof=1).values
    sigma2 = c_step + d_step * spread2
    sigma2 = np.maximum(sigma2, 1e-6)
    sigma = np.sqrt(sigma2)

    pred_df.loc[y_next.index, "mu"] = mu[0]
    pred_df.loc[y_next.index, "sigma"] = sigma[0]
    pred_df.loc[y_next.index, "sigma2"] = sigma2[0]
    pred_df.loc[y_next.index, "obs"] = y_true_test[0]

    # add newly observed row to history for next step
    X_hist = pd.concat([X_hist, X_next])
    y_hist = pd.concat([y_hist, y_next])

pred_df["pit"] = norm.cdf(pred_df.obs, loc=pred_df.mu, scale=pred_df.sigma)

In [ ]:
pred_df.dropna().pit.plot(kind='hist', bins=20, density=True)